# Text-to-SVG Pipeline: Training + Inference + Pruning
**NYU DL Spring 2026 — Kaggle Contest**

This notebook covers everything end-to-end:
1. Environment setup
2. Data loading + quality filtering
3. Model training (1.5B or 7B, switchable)
4. vLLM inference with full recovery pipeline
5. Magnitude pruning + score tradeoff analysis
6. Submission generation

**GPU recommendation:**
- 1.5B model: single A100 80GB
- 7B model: single A100 80GB (fits at 4-bit)
- Run multiple instances in parallel with different `MODEL_CHOICE` values

## 0. Install dependencies

In [ ]:
# Run once per runtime
import subprocess, sys

pkgs = [
    'unsloth[colab-new]',
    'vllm',
    'datasets',
    'trl',
    'transformers>=4.40.0',
    'accelerate',
    'peft',
    'bitsandbytes',
    'pandas',
    'cairosvg',   # for local SVG rendering validation
    'lxml',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs)
print('✅ Packages installed')

## 1. Global config — change MODEL_CHOICE to switch experiments

In [ ]:
# ============================================================
# MASTER SWITCH — change this to run different experiments
# Options: '1.5B' | '7B' | 'deepseek-6.7B'
# ============================================================
MODEL_CHOICE = '7B'

# ============================================================
# PATHS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT    = '/content/drive/MyDrive/DL_SVG'
LORA_SAVE_DIR = f'{DRIVE_ROOT}/lora_{MODEL_CHOICE.replace(".","_")}'
MERGED_DIR    = f'/content/merged_{MODEL_CHOICE.replace(".","_")}'
TEST_CSV      = '/content/test.csv'
SUBMISSION    = f'{DRIVE_ROOT}/submission_{MODEL_CHOICE.replace(".","_")}.csv'
CLEAN_DATA    = f'{DRIVE_ROOT}/clean_training_data.parquet'

os.makedirs(DRIVE_ROOT, exist_ok=True)

# ============================================================
# MODEL CONFIGS
# ============================================================
MODEL_CONFIGS = {
    '1.5B': {
        'model_name':                    'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit',
        'max_seq_length':                3072,
        'lora_r':                        32,
        'lora_alpha':                    64,
        'learning_rate':                 3e-4,
        'num_train_epochs':              4,
        'per_device_train_batch_size':   16,
        'gradient_accumulation_steps':   2,
        'warmup_ratio':                  0.05,
        'weight_decay':                  0.01,
    },
    '7B': {
        'model_name':                    'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit',
        'max_seq_length':                3072,
        'lora_r':                        32,
        'lora_alpha':                    64,
        'learning_rate':                 2e-4,   # lower LR for larger model
        'num_train_epochs':              3,
        'per_device_train_batch_size':   4,      # fits in 80GB at 4bit
        'gradient_accumulation_steps':   8,      # effective batch = 32
        'warmup_ratio':                  0.05,
        'weight_decay':                  0.01,
    },
    'deepseek-6.7B': {
        'model_name':                    'unsloth/deepseek-coder-6.7b-instruct-bnb-4bit',
        'max_seq_length':                3072,
        'lora_r':                        32,
        'lora_alpha':                    64,
        'learning_rate':                 2e-4,
        'num_train_epochs':              3,
        'per_device_train_batch_size':   4,
        'gradient_accumulation_steps':   8,
        'warmup_ratio':                  0.05,
        'weight_decay':                  0.01,
    },
}

CFG = MODEL_CONFIGS[MODEL_CHOICE]
CFG.update({
    'lora_save_dir':  LORA_SAVE_DIR,
    'merged_dir':     MERGED_DIR,
    'logging_steps':  20,
    'eval_steps':     200,
    'save_steps':     400,
    'eval_size':      0.02,
    'seed':           42,
})

print(f'Model:         {CFG["model_name"]}')
print(f'LoRA save dir: {LORA_SAVE_DIR}')
print(f'Merged dir:    {MERGED_DIR}')

## 2. Seed + imports

In [ ]:
import os, re, gc, time, json, random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset

SEED = CFG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Data loading + quality filtering

In [ ]:
# ============================================================
# SVG QUALITY GATE
# ============================================================
# These filters address the root causes we identified:
#   - filling="0" artifact from SVGX dialect
#   - empty fill attributes (renders invisible)
#   - degenerate zero-area repeated paths
#   - wrong canvas size
#   - too short/long

ET.register_namespace("", "http://www.w3.org/2000/svg")
ET.register_namespace("xlink", "http://www.w3.org/1999/xlink")

DISALLOWED_ELEMENTS = {'script', 'foreignObject', 'animate', 'animateTransform',
                       'animateMotion', 'set', 'discard'}

def is_clean_svg(svg: str) -> bool:
    """Returns True only for high-quality, competition-compatible SVGs."""
    if not svg or not isinstance(svg, str):
        return False
    svg = svg.strip()
    if not svg.lower().startswith('<svg'):
        return False
    # Length bounds (competition max is 16000)
    if len(svg) < 80 or len(svg) > 14000:
        return False
    # Reject SVGX dialect artifacts
    if 'filling=' in svg:
        return False
    # Reject empty fill (renders as transparent black outline, looks wrong)
    if 'fill=""' in svg or "fill=''" in svg:
        return False
    # Reject disallowed elements from competition spec
    for el in DISALLOWED_ELEMENTS:
        if f'<{el}' in svg.lower():
            return False
    # Must parse as valid XML
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return False
    children = list(root)
    if len(children) == 0:
        return False
    # Reject if majority of children are identical (degenerate loop pattern)
    if len(children) >= 4:
        canonical = [tuple(sorted(c.attrib.items())) for c in children]
        most_common_count = max(Counter(canonical).values())
        if most_common_count / len(canonical) > 0.75:
            return False
    # Must contain at least one path/shape with non-trivial geometry
    has_real_geometry = False
    for el in root.iter():
        d = el.get('d', '')
        if d:
            coords = set(re.findall(r'-?\d+\.?\d*', d))
            if len(coords) >= 5:  # at least 5 distinct numeric values
                has_real_geometry = True
                break
        # Also accept rects/circles with real dimensions
        if el.tag.split('}')[-1] in ('rect', 'circle', 'ellipse', 'polygon'):
            has_real_geometry = True
            break
    return has_real_geometry


def normalize_svg(svg: str) -> str:
    """Normalize SVG to match competition format before training."""
    # Standardize to competition-required 256x256 canvas
    svg = re.sub(r'viewBox=["\']0 0 \d+ \d+["\']', 'viewBox="0 0 256 256"', svg)
    svg = re.sub(r'(width|height)=["\']\d+(?:px)?["\']',
                 lambda m: m.group(0).split('=')[0] + '="256"', svg)
    # Remove SVGX artifacts
    svg = re.sub(r'\s*filling="[^"]*"', '', svg)
    svg = re.sub(r'\s*fill-opacity="1"', '', svg)  # redundant default
    # Ensure xmlns is present
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg.strip()


def pick_field(row: dict, fields: list) -> str:
    for f in fields:
        if f in row and row[f] is not None:
            v = str(row[f]).strip()
            if v:
                return v
    return ''


print('Quality gate functions defined')

In [ ]:
# ============================================================
# DATASET SOURCES
# ============================================================
# Priority order: cleanest sources first
# starvector/svg-icons — standard SVG, no artifacts, icon-focused
# OmniSVG/MMSVG-Icon  — use picosvg field (cleaned version)
# SVGX_SFT_GEN_51k    — use the CURATED subset, not the full 1M
# thesantatitan       — deepseek-generated, generally clean

SOURCES = [
    {
        'id':     'starvector/svg-icons',
        'split':  'train',
        'kwargs': {},
        'prompt_fields': ['description', 'name', 'caption'],
        'svg_fields':    ['svg'],
        'max_samples':   30000,
    },
    {
        'id':     'OmniSVG/MMSVG-Icon',
        'split':  'train',
        'kwargs': {},
        'prompt_fields': ['description', 'prompt', 'keywords'],
        'svg_fields':    ['picosvg', 'svg'],  # picosvg is the cleaned version
        'max_samples':   30000,
    },
    {
        'id':     'xingxm/SVGX-SFT-1M',
        'split':  'train',
        # NOTE: use the curated 51k subset file explicitly
        'kwargs': {'data_files': 'SVGX_SFT_GEN_51k.json'},
        'prompt_fields': ['prompt', 'instruction', 'input'],
        'svg_fields':    ['completion', 'output', 'svg'],
        'max_samples':   25000,
    },
    {
        'id':     'thesantatitan/deepseek-svg-dataset',
        'split':  'train',
        'kwargs': {},
        'prompt_fields': ['prompt', 'instruction', 'input'],
        'svg_fields':    ['completion', 'output', 'svg'],
        'max_samples':   15000,
    },
]


def load_and_filter_source(src: dict) -> list:
    print(f"\nLoading {src['id']}...")
    try:
        ds = load_dataset(src['id'], split=src['split'], **src['kwargs'])
    except Exception as e:
        print(f'  ERROR: {e}')
        return []

    if src['max_samples'] and len(ds) > src['max_samples']:
        ds = ds.shuffle(seed=SEED).select(range(src['max_samples']))

    rows, rejected = [], 0
    for row in ds:
        prompt = pick_field(row, src['prompt_fields'])
        svg    = pick_field(row, src['svg_fields'])
        if not prompt or not svg:
            rejected += 1
            continue
        if not is_clean_svg(svg):
            rejected += 1
            continue
        rows.append({'prompt': prompt, 'svg': normalize_svg(svg)})

    print(f'  Accepted: {len(rows):,}  Rejected: {rejected:,}  '
          f'({rejected/(len(rows)+rejected)*100:.1f}% filtered)')
    return rows


# Check if we already have clean data saved
if os.path.exists(CLEAN_DATA):
    print(f'Loading cached clean data from {CLEAN_DATA}')
    clean_df = pd.read_parquet(CLEAN_DATA)
    print(f'Total clean rows: {len(clean_df):,}')
else:
    all_rows = []
    for src in SOURCES:
        all_rows.extend(load_and_filter_source(src))

    clean_df = pd.DataFrame(all_rows).drop_duplicates(subset='prompt').reset_index(drop=True)
    clean_df.to_parquet(CLEAN_DATA, index=False)
    print(f'\nTotal clean rows: {len(clean_df):,}')
    print(f'Saved to {CLEAN_DATA}')

print('\nSample prompts:')
for p in clean_df['prompt'].sample(5, random_state=SEED).tolist():
    print(f'  • {p[:80]}')

## 4. Build training dataset with curriculum ordering

In [ ]:
# ============================================================
# CURRICULUM LEARNING: sort by SVG complexity (easy → hard)
# Simple SVGs (fewer paths, shorter) are trained first.
# This gives the model a foundation before complex shapes.
# ============================================================

def svg_complexity(svg: str) -> int:
    """Proxy for visual complexity: number of distinct path commands."""
    return len(re.findall(r'<(?:path|rect|circle|ellipse|polygon|polyline)', svg))

clean_df['complexity'] = clean_df['svg'].apply(svg_complexity)
clean_df['svg_len']    = clean_df['svg'].apply(len)

print('Complexity distribution:')
print(clean_df['complexity'].describe())

# Sort easy → hard for curriculum training
# We'll use a mild sort: not strictly sorted, but complexity-weighted
clean_df = clean_df.sort_values('complexity').reset_index(drop=True)

print(f'\nMin complexity: {clean_df["complexity"].min()}')
print(f'Max complexity: {clean_df["complexity"].max()}')
print(f'Median:         {clean_df["complexity"].median()}')

In [ ]:
# ============================================================
# FORMAT INTO CHATML TRAINING FORMAT
# ============================================================

SYSTEM_PROMPT = (
    "You generate valid SVG markup from user requests. "
    "Return ONLY the SVG element with no explanation. "
    "Requirements: xmlns='http://www.w3.org/2000/svg', "
    "width='256', height='256', viewBox='0 0 256 256'. "
    "Use only standard SVG attributes."
)


def format_training_example(row: dict) -> dict:
    """Format a row into the ChatML instruction format."""
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['svg']}</svg><|im_end|>"
    )
    return {'text': text}


records = [format_training_example(r) for r in clean_df.to_dict('records')]
full_ds = Dataset.from_list(records)

# Train/eval split
split   = full_ds.train_test_split(test_size=CFG['eval_size'], seed=SEED)
train_ds = split['train']
eval_ds  = split['test']

print(f'Train: {len(train_ds):,}  Eval: {len(eval_ds):,}')
print('\nSample training text (first 300 chars):')
print(train_ds[0]['text'][:300])

## 5. Load model + apply LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = CFG['model_name'],
    max_seq_length= CFG['max_seq_length'],
    dtype         = torch.bfloat16,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                   = CFG['lora_r'],
    lora_alpha          = CFG['lora_alpha'],
    lora_dropout        = 0,       # optimized for Unsloth
    bias                = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves 30% VRAM
    random_state        = SEED,
    use_rslora          = True,    # rank-stabilized LoRA, better at large r
    target_modules      = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

print(model.print_trainable_parameters())

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir                  = '/content/checkpoints',
    num_train_epochs            = CFG['num_train_epochs'],
    per_device_train_batch_size = CFG['per_device_train_batch_size'],
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = CFG['gradient_accumulation_steps'],
    warmup_ratio                = CFG['warmup_ratio'],
    learning_rate               = CFG['learning_rate'],
    lr_scheduler_type           = 'cosine',
    weight_decay                = CFG['weight_decay'],
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    optim                       = 'adamw_8bit',
    logging_steps               = CFG['logging_steps'],
    eval_strategy               = 'steps',
    eval_steps                  = CFG['eval_steps'],
    save_strategy               = 'steps',
    save_steps                  = CFG['save_steps'],
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    seed                        = SEED,
    report_to                   = 'none',
    dataloader_num_workers      = 4,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    dataset_text_field = 'text',
    max_seq_length  = CFG['max_seq_length'],
    dataset_num_proc= 4,
    args            = training_args,
)

print(f'Steps per epoch: {len(train_ds) // (CFG["per_device_train_batch_size"] * CFG["gradient_accumulation_steps"])}')
print(f'Total steps:     {trainer.args.max_steps if trainer.args.max_steps > 0 else "auto"}')

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} min')

In [ ]:
# Save LoRA weights to Drive
print(f'Saving LoRA to {LORA_SAVE_DIR}...')
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)
print('Saved.')

## 7. Merge LoRA + export for vLLM

In [ ]:
if not os.path.exists(MERGED_DIR):
    print(f'Merging LoRA weights into {MERGED_DIR}...')
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')

    # Free GPU memory before vLLM claims it
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print('Merge complete. GPU memory cleared.')
else:
    print(f'Merged model already exists at {MERGED_DIR}. Skipping.')
    try:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    except:
        pass

## 8. Post-processing pipeline

In [ ]:
# ============================================================
# All post-processing functions in one cell for easy reference
# ============================================================

ET.register_namespace("", "http://www.w3.org/2000/svg")
ET.register_namespace("xlink", "http://www.w3.org/1999/xlink")

SVG_FULL_RE = re.compile(r'<svg[\s\S]*?</svg>', re.IGNORECASE)


def truncate_at_loop(text: str) -> tuple:
    """Detect and truncate repetition loops in raw generated text."""
    match = re.search(r'(.{30,}?)\1{3,}', text)
    if match:
        return text[:match.start() + len(match.group(1))], True
    return text, False


def extract_svg(text: str) -> str:
    m = SVG_FULL_RE.search(text)
    return m.group(0).strip() if m else ''


def repair_svg(text: str) -> str | None:
    """Recover truncated SVG (primary path since stop string strips </svg>)."""
    svg_start = text.lower().find('<svg')
    if svg_start == -1:
        return None
    text = text[svg_start:].strip()
    if re.search(r'</svg\s*>', text, re.IGNORECASE):
        return text
    last_boundary = max(text.rfind('/>'), text.rfind('>'))
    if last_boundary == -1:
        return None
    text = text[:last_boundary + 1]
    VOID_TAGS = {'circle','ellipse','line','path','polygon','polyline',
                 'rect','use','image','stop'}
    open_tags = []
    for m in re.finditer(r'<(/?)(\w[\w:_-]*)([^>]*?)(/?)>', text):
        is_close   = m.group(1) == '/'
        tag        = m.group(2).lower()
        self_close = m.group(4) == '/'
        if tag == 'svg':
            if not is_close:
                open_tags = []
            continue
        if self_close or tag in VOID_TAGS:
            continue
        if is_close:
            if open_tags and open_tags[-1] == tag:
                open_tags.pop()
        else:
            open_tags.append(tag)
    closing = ''.join(f'</{t}>' for t in reversed(open_tags))
    return text + closing + '</svg>'


def is_valid_svg(svg_text: str) -> bool:
    if not svg_text:
        return False
    try:
        return ET.fromstring(svg_text).tag.split('}')[-1] == 'svg'
    except ET.ParseError:
        return False


def is_degenerate_path(d_attr: str) -> bool:
    """Zero-area path: all coordinates are identical."""
    coords = re.findall(r'-?\d+\.?\d*', d_attr)
    return len(coords) > 2 and len(set(coords)) <= 2


def deduplicate_svg(svg_text: str) -> tuple:
    """Remove duplicate/degenerate children. Always keeps at least 1 child."""
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return svg_text, 0
    children = list(root)
    if not children:
        return svg_text, 0
    seen, to_remove = set(), []
    for child in children:
        tag_local = child.tag.split('}')[-1]
        if tag_local == 'path' and is_degenerate_path(child.get('d', '')):
            to_remove.append(child)
            continue
        canonical = (child.tag, tuple(sorted(child.attrib.items())))
        if canonical in seen:
            to_remove.append(child)
        else:
            seen.add(canonical)
    # Never remove the last remaining child
    to_remove = to_remove[:max(0, len(children) - 1)]
    for child in to_remove:
        root.remove(child)
    return ET.tostring(root, encoding='unicode'), len(to_remove)


def clean_svg_attributes(svg: str) -> str:
    """Remove artifacts that hurt visual quality."""
    svg = re.sub(r'\s*filling="[^"]*"', '', svg)     # SVGX artifact
    svg = re.sub(r'fill=""', 'fill="#000000"', svg)   # empty -> black
    svg = re.sub(r"fill=''", 'fill="#000000"', svg)
    svg = re.sub(r'\s*fill-opacity="1"', '', svg)      # redundant default
    return svg


def normalize_canvas(svg: str) -> str:
    """Force competition 256x256 canvas — critical for SSIM scoring."""
    svg = re.sub(r'viewBox=["\']\s*0\s+0\s+\d+\s+\d+["\']', 'viewBox="0 0 256 256"', svg)
    svg = re.sub(r'(?<![\w-])height=["\']\d+(?:px)?["\']', 'height="256"', svg)
    svg = re.sub(r'(?<![\w-])width=["\']\d+(?:px)?["\']',  'width="256"',  svg)
    if 'viewBox' not in svg:
        svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    if 'width=' not in svg.split('>')[0]:
        svg = svg.replace('<svg', '<svg width="256" height="256"', 1)
    return svg


def optimize_svg(svg: str) -> str:
    """Compact the SVG string without changing visual output."""
    svg = re.sub(r'[\n\r\t]', '', svg)
    svg = re.sub(r'>\s+<', '><', svg)
    svg = re.sub(r'\s*=\s*', '=', svg)
    svg = re.sub(r'\s+', ' ', svg)
    def _round(m):
        v = float(m.group(0))
        return str(int(v)) if v == int(v) else f'{v:.1f}'
    return re.sub(r'-?\d+\.\d+', _round, svg).strip()


def fallback_svg(_prompt: str = '') -> str:
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def process_output(generated_text: str, prompt: str = '') -> tuple:
    """
    Full 5-stage recovery pipeline.
    Returns (svg_string, stage_label).
    """
    # Stage 0: Truncate loops in raw text BEFORE any extraction
    cleaned, was_looped = truncate_at_loop(generated_text)

    # Stage 1: Try complete SVG in output
    svg = extract_svg(cleaned)
    if not is_valid_svg(svg):
        # Stage 2: Repair truncated SVG (normal path; stop string strips </svg>)
        svg = repair_svg(cleaned)
        if not svg or not is_valid_svg(svg):
            return fallback_svg(prompt), 'fallback'
        stage = 'loop_repaired' if was_looped else 'repaired'
    else:
        stage = 'loop_direct' if was_looped else 'direct'

    # Stage 3: Deduplicate identical/degenerate elements
    svg, n_removed = deduplicate_svg(svg)
    if not svg:
        return fallback_svg(prompt), 'fallback'
    if n_removed > 0:
        stage += '_deduped'

    # Stage 4: Clean artifacts + normalize canvas + compact
    svg = clean_svg_attributes(svg)
    svg = normalize_canvas(svg)
    svg = optimize_svg(svg)

    # Final safety check
    if not is_valid_svg(svg):
        return fallback_svg(prompt), 'fallback_post_clean'

    return svg, stage


print('Post-processing pipeline loaded')

## 9. vLLM inference

In [ ]:
from vllm import LLM, SamplingParams

print(f'Booting vLLM from {MERGED_DIR}...')
llm = LLM(
    model          = MERGED_DIR,
    max_model_len  = 3072,
    tensor_parallel_size = 1,
    gpu_memory_utilization = 0.92,
    disable_log_stats = True,
)
print('vLLM ready.')

In [ ]:
# ============================================================
# SAMPLING PARAMS
# Two modes: fast greedy (competition) vs best-of-N (higher quality)
# ============================================================

# NOTE: stop=["</svg>"] strips the closing tag from output.
# This is intentional — repair_svg() is designed to handle it
# and appends </svg> back. This prevents the model from looping
# past the closing tag.
# NO repetition_penalty or frequency_penalty — SVG is structurally
# repetitive and any token penalty destroys generation quality.

BEST_OF_N = 3   # set to 1 for fastest, 3 for best quality

sampling_params = SamplingParams(
    temperature = 0.0 if BEST_OF_N == 1 else 0.4,
    max_tokens  = 2500,
    n           = BEST_OF_N,
    stop        = ['</svg>', '<|im_end|>', '<|endoftext|>'],
)

print(f'Sampling mode: {"greedy" if BEST_OF_N == 1 else f"best-of-{BEST_OF_N} @ temp=0.4"}')

In [ ]:
# ============================================================
# RUN INFERENCE
# ============================================================

print(f'Reading {TEST_CSV}...')
test_df  = pd.read_csv(TEST_CSV)
prompts  = test_df['prompt'].tolist()
ids      = test_df['id'].tolist()

chat_texts = [
    f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
    f'<|im_start|>user\n{p}<|im_end|>\n'
    f'<|im_start|>assistant\n'
    for p in prompts
]

print(f'Running inference on {len(chat_texts):,} prompts...')
t0 = time.time()
outputs = llm.generate(chat_texts, sampling_params)

# ============================================================
# POST-PROCESS OUTPUTS
# ============================================================

rows  = []
stats = {}

for i, output in enumerate(outputs):
    if BEST_OF_N == 1:
        svg, stage = process_output(output.outputs[0].text, prompts[i])
    else:
        # Best-of-N: pick shortest valid SVG (best compactness score)
        candidates = []
        for candidate in output.outputs:
            svg_c, stage_c = process_output(candidate.text, prompts[i])
            if 'fallback' not in stage_c:
                candidates.append((svg_c, stage_c, len(svg_c)))
        if candidates:
            svg, stage, _ = min(candidates, key=lambda x: x[2])
        else:
            svg, stage = fallback_svg(prompts[i]), 'fallback'

    stats[stage] = stats.get(stage, 0) + 1
    rows.append({'id': ids[i], 'svg': svg})

# Save submission
os.makedirs(os.path.dirname(SUBMISSION), exist_ok=True)
pd.DataFrame(rows).to_csv(SUBMISSION, index=False)

total   = len(rows)
elapsed = (time.time() - t0) / 60
fallback_total = sum(v for k, v in stats.items() if 'fallback' in k)

print(f'\n✅ Saved to {SUBMISSION}  ({elapsed:.2f} min)')
print(f'\n--- Output Quality Breakdown ---')
for stage, count in sorted(stats.items(), key=lambda x: -x[1]):
    bar = '█' * int(count / total * 40)
    print(f'  {stage:<28} {count:>5}  ({count/total*100:5.1f}%)  {bar}')
print(f'\n  Total non-fallback: {total - fallback_total} ({(total-fallback_total)/total*100:.1f}%)')
print(f'  Total fallback:     {fallback_total} ({fallback_total/total*100:.1f}%)')

## 10. Pruning experiment
Run this section AFTER you have your best model.
Prune at 10/20/30% sparsity, measure score impact.
Document the tradeoff table in your report.

In [ ]:
# ============================================================
# MAGNITUDE PRUNING EXPERIMENT
# Prune a copy of the merged model at multiple sparsity levels
# and evaluate on a small validation subset to measure score drop.
# ============================================================

import copy
from torch.nn.utils import prune
from transformers import AutoModelForCausalLM, AutoTokenizer

PRUNING_LEVELS   = [0.10, 0.20, 0.30, 0.40]
PRUNING_EVAL_N   = 200   # prompts to evaluate per level (speed vs accuracy)
PRUNING_RESULTS  = []


def apply_magnitude_pruning(model: torch.nn.Module, sparsity: float) -> torch.nn.Module:
    """
    Global unstructured L1 magnitude pruning.
    Prunes the lowest-magnitude weights across ALL linear layers.
    This is the standard baseline — structured pruning would be
    better for inference speed but requires more effort.
    """
    params_to_prune = [
        (module, 'weight')
        for module in model.modules()
        if isinstance(module, torch.nn.Linear)
    ]
    prune.global_unstructured(
        params_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=sparsity,
    )
    # Make pruning permanent (remove the _orig buffers)
    for module, _ in params_to_prune:
        prune.remove(module, 'weight')
    return model


def count_zeros(model: torch.nn.Module) -> float:
    """Measure actual sparsity achieved."""
    total, zeros = 0, 0
    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            total += module.weight.numel()
            zeros += (module.weight == 0).sum().item()
    return zeros / total if total > 0 else 0


def quick_eval_svg_quality(model_path: str, prompts_subset: list,
                           tokenizer_path: str) -> dict:
    """
    Quick quality evaluation without vLLM.
    Uses HuggingFace generate() for small eval batches.
    Returns: {valid_rate, avg_complexity, fallback_rate}
    """
    from transformers import pipeline
    pipe = pipeline('text-generation', model=model_path,
                    tokenizer=tokenizer_path,
                    torch_dtype=torch.bfloat16,
                    device_map='auto')

    inputs = [
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{p}<|im_end|>\n'
        f'<|im_start|>assistant\n'
        for p in prompts_subset
    ]

    results = pipe(inputs, max_new_tokens=800, temperature=0.0,
                   do_sample=False, batch_size=4)

    valid, fallback_count, complexities = 0, 0, []
    for result, prompt in zip(results, prompts_subset):
        text = result[0]['generated_text']
        svg, stage = process_output(text, prompt)
        if 'fallback' in stage:
            fallback_count += 1
        else:
            valid += 1
            complexities.append(svg_complexity(svg))

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'valid_rate':      valid / len(prompts_subset),
        'fallback_rate':   fallback_count / len(prompts_subset),
        'avg_complexity':  float(np.mean(complexities)) if complexities else 0,
    }


print('Pruning functions defined.')
print('Run the next cell to execute pruning experiments.')

In [ ]:
# ============================================================
# RUN PRUNING EXPERIMENTS
# Loads a fresh copy of the merged model for each sparsity level.
# NOTE: this section requires significant RAM. Run on A100 80GB.
# ============================================================

eval_prompts = prompts[:PRUNING_EVAL_N]

# Baseline (no pruning)
print('Evaluating baseline (0% pruning)...')
baseline = quick_eval_svg_quality(MERGED_DIR, eval_prompts, MERGED_DIR)
PRUNING_RESULTS.append({'sparsity': 0.0, 'actual_sparsity': 0.0, **baseline})
print(f'  Baseline: valid={baseline["valid_rate"]:.1%}  '
      f'fallback={baseline["fallback_rate"]:.1%}  '
      f'avg_complexity={baseline["avg_complexity"]:.1f}')

for sparsity in PRUNING_LEVELS:
    print(f'\nPruning at {sparsity:.0%}...')

    # Load a fresh copy of the model (don't reuse the pruned one)
    prune_model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=torch.bfloat16, device_map='auto')

    # Apply pruning
    prune_model = apply_magnitude_pruning(prune_model, sparsity)
    actual_sparsity = count_zeros(prune_model)
    print(f'  Actual sparsity: {actual_sparsity:.1%}')

    # Save pruned model to a temp path
    pruned_path = f'/content/pruned_{int(sparsity*100)}pct'
    prune_model.save_pretrained(pruned_path)
    AutoTokenizer.from_pretrained(MERGED_DIR).save_pretrained(pruned_path)

    del prune_model
    gc.collect()
    torch.cuda.empty_cache()

    # Evaluate
    metrics = quick_eval_svg_quality(pruned_path, eval_prompts, pruned_path)
    PRUNING_RESULTS.append({
        'sparsity': sparsity,
        'actual_sparsity': actual_sparsity,
        **metrics
    })
    print(f'  valid={metrics["valid_rate"]:.1%}  '
          f'fallback={metrics["fallback_rate"]:.1%}  '
          f'avg_complexity={metrics["avg_complexity"]:.1f}')

# ============================================================
# REPORT TABLE
# ============================================================
results_df = pd.DataFrame(PRUNING_RESULTS)
results_df['valid_drop'] = results_df['valid_rate'].iloc[0] - results_df['valid_rate']

print('\n' + '='*65)
print('PRUNING RESULTS (for report)')
print('='*65)
print(results_df[['sparsity','actual_sparsity','valid_rate',
                   'fallback_rate','avg_complexity','valid_drop']].to_string(index=False))

# Save results to Drive
results_df.to_csv(f'{DRIVE_ROOT}/pruning_results_{MODEL_CHOICE}.csv', index=False)
print(f'\nSaved to {DRIVE_ROOT}/pruning_results_{MODEL_CHOICE}.csv')

# Recommendation
# Find the highest sparsity with < 2% valid_rate drop
safe = results_df[results_df['valid_drop'] < 0.02]
if len(safe) > 1:
    best = safe.iloc[-1]
    print(f'\n✅ Recommendation: {best["sparsity"]:.0%} pruning keeps valid_rate '
          f'within 2% of baseline ({best["valid_rate"]:.1%} vs '
          f'{results_df.iloc[0]["valid_rate"]:.1%})')

## 11. Hyperparameter sweep helper
Run this to test multiple LoRA configurations quickly.
Uses the best model config found from your hyperparameter screenshot.

In [ ]:
# ============================================================
# QUICK SWEEP: compare LoRA configs on small training subset
# From your hyperparameter sweep results, best was:
#   lora_r=32, lora_alpha=64, lr=3e-4 → lowest train_loss
# This cell tests whether higher r (64/128) improves on clean data
# ============================================================

SWEEP_CONFIGS = [
    # Best from your previous sweep
    {'lora_r': 32,  'lora_alpha': 64,  'learning_rate': 3e-4,  'label': 'r32-a64-lr3e4'},
    # Try higher rank (more capacity, clean data may benefit)
    {'lora_r': 64,  'lora_alpha': 128, 'learning_rate': 2e-4,  'label': 'r64-a128-lr2e4'},
    # Try lower LR with higher rank (more stable)
    {'lora_r': 64,  'lora_alpha': 64,  'learning_rate': 1e-4,  'label': 'r64-a64-lr1e4'},
]

SWEEP_TRAIN_SIZE = 5000   # small subset for fast sweep
SWEEP_EPOCHS     = 2

sweep_ds = Dataset.from_list(
    [format_training_example(r)
     for r in clean_df.sample(SWEEP_TRAIN_SIZE, random_state=SEED).to_dict('records')]
)
sweep_results = []


def run_sweep_config(sweep_cfg: dict) -> dict:
    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments

    print(f"\n--- Sweep: {sweep_cfg['label']} ---")

    m, tok = FastLanguageModel.from_pretrained(
        CFG['model_name'], max_seq_length=2048,
        dtype=torch.bfloat16, load_in_4bit=True)

    m = FastLanguageModel.get_peft_model(
        m, r=sweep_cfg['lora_r'],
        lora_alpha=sweep_cfg['lora_alpha'],
        lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED, use_rslora=True,
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
    )

    args = TrainingArguments(
        output_dir=f'/content/sweep_{sweep_cfg["label"]}',
        num_train_epochs=SWEEP_EPOCHS,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        learning_rate=sweep_cfg['learning_rate'],
        lr_scheduler_type='cosine',
        bf16=True, optim='adamw_8bit',
        logging_steps=20, report_to='none',
    )

    trainer = SFTTrainer(
        model=m, tokenizer=tok,
        train_dataset=sweep_ds,
        dataset_text_field='text',
        max_seq_length=2048, args=args,
    )

    trainer.train()
    final_loss = trainer.state.log_history[-1].get('train_loss', float('nan'))

    del m, tok, trainer
    gc.collect()
    torch.cuda.empty_cache()

    result = {**sweep_cfg, 'train_loss': final_loss}
    print(f"  train_loss: {final_loss:.4f}")
    return result


print('Sweep defined. Uncomment the loop below to run.')
print('Each config takes ~15-20min on A100.')

# Uncomment to run:
# for cfg in SWEEP_CONFIGS:
#     sweep_results.append(run_sweep_config(cfg))
# print(pd.DataFrame(sweep_results).sort_values('train_loss'))

## 12. Quick sanity check — render a few SVGs

In [ ]:
# View 10 sample outputs from the submission directly in the notebook
from IPython.display import display, HTML
import html as htmllib

sub_df = pd.read_csv(SUBMISSION)

sample = sub_df.sample(10, random_state=SEED)
cells = []
for _, row in sample.iterrows():
    prompt_escaped = htmllib.escape(str(test_df.loc[test_df['id']==row['id'], 'prompt'].values[0]
                                    if row['id'] in test_df['id'].values else '')[:60])
    cells.append(
        f'<td style="padding:8px;text-align:center;vertical-align:top;">'
        f'<div style="font-size:11px;color:#666;margin-bottom:4px;">{prompt_escaped}</div>'
        f'{row["svg"]}'
        f'</td>'
    )

rows_html = ''.join(
    f'<tr>{cells[i]}{cells[i+1] if i+1 < len(cells) else "<td></td>"}</tr>'
    for i in range(0, len(cells), 2)
)
display(HTML(f'<table style="border-collapse:collapse;">{rows_html}</table>'))

## 13. Summary stats for report

In [ ]:
# Generate the stats table you'll paste into your report

sub_df = pd.read_csv(SUBMISSION)
svg_lengths = sub_df['svg'].apply(len)
path_counts = sub_df['svg'].apply(svg_complexity)

valid_count = sub_df['svg'].apply(lambda s: 1 if is_valid_svg(s) else 0).sum()

print('=' * 50)
print('SUBMISSION STATS (paste into report)')
print('=' * 50)
print(f'Model:              {MODEL_CHOICE}')
print(f'Total submissions:  {len(sub_df):,}')
print(f'Valid SVGs:         {valid_count:,} ({valid_count/len(sub_df)*100:.1f}%)')
print(f'Avg SVG length:     {svg_lengths.mean():.0f} chars')
print(f'Median SVG length:  {svg_lengths.median():.0f} chars')
print(f'Max SVG length:     {svg_lengths.max()} chars')
print(f'Avg path count:     {path_counts.mean():.1f}')
print(f'Max path count:     {path_counts.max()}')
print()

# Check canvas compliance
canvas_256 = sub_df['svg'].str.contains('256 256').sum()
print(f'Canvas 256x256:     {canvas_256:,} ({canvas_256/len(sub_df)*100:.1f}%)')

# Check for artifacts
has_filling = sub_df['svg'].str.contains('filling=').sum()
has_empty_fill = sub_df['svg'].str.contains('fill=""').sum()
print(f'filling= artifacts: {has_filling}')
print(f'empty fill:         {has_empty_fill}')